# FerrisRes: Gemma-4 Distillation on Colab T4

Distill Gemma-4 models into FerrisRes Block AttnRes format.

**Supported models (T4 / 16GB VRAM):**
- **E2B** (~4.6 GB FP16, 35 layers, dense) — fits on free T4

**Too large for T4 (requires A100 80GB):**
- **E4B** (~9 GB) — OOMs on free T4, needs Colab Pro
- **26B-A4B** (~52 GB FP16, MoE-128)
- **31B** (~62 GB FP16, dense)

The `.deb` package installs the FerrisRes binary, Vulkan dependencies, and the
NVIDIA ICD file — everything needed for GPU inference on Colab in one step.

**Prerequisites:** Runtime → Change runtime type → **T4 GPU**

In [ ]:
#@title Configuration
model_config = "e2b" #@param ["e2b", "e4b"]
learning_rate = 0.00004 #@param {type:"number"}
seq_len = 256 #@param {type:"integer"}
max_steps = 10000 #@param {type:"integer"}
converge_threshold = 0.001 #@param {type:"number"}
converge_patience = 150 #@param {type:"integer"}

print(f"Config: {model_config}, seq_len={seq_len}, lr={learning_rate}, max_steps={max_steps}")

In [ ]:
# 1. Install FerrisRes via .deb (binary + Vulkan deps + NVIDIA ICD)
import os, requests

REPO = "shift/FerrisRes"
installed = False

# Try downloading .deb from latest GitHub release
try:
    api = f"https://api.github.com/repos/{REPO}/releases/latest"
    release = requests.get(api, timeout=10).json()
    for asset in release.get('assets', []):
        if asset['name'].endswith('_amd64.deb'):
            print(f"Downloading {asset['name']} from {release['tag_name']}...")
            r = requests.get(asset['browser_download_url'], timeout=60)
            with open('ferrisres.deb', 'wb') as f:
                f.write(r.content)
            !dpkg -i ferrisres.deb 2>&1 | tail -5
            !rm ferrisres.deb
            print(f"Installed FerrisRes {release['tag_name']} via .deb")
            installed = True
            break
except Exception as e:
    print(f"Release download failed: {e}")

if not installed:
    # Fallback: try the CI artifact from latest main build
    print("No .deb in release. Trying CI artifact...")
    try:
        runs = requests.get(
            f"https://api.github.com/repos/{REPO}/actions/runs?branch=main&status=success&per_page=1",
            timeout=10
        ).json()
        if runs.get('workflow_runs'):
            run_id = runs['workflow_runs'][0]['id']
            artifacts = requests.get(
                f"https://api.github.com/repos/{REPO}/actions/runs/{run_id}/artifacts",
                timeout=10
            ).json()
            for art in artifacts.get('artifacts', []):
                if 'colab' in art['name']:
                    print(f"CI artifact requires auth. Falling back to source compile.")
                    break
    except Exception:
        pass

if not installed:
    # Last resort: compile from source (~9 min on T4)
    print("Compiling from source (~9 min on T4)...")
    !curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y 2>&1 | tail -1
    os.environ['PATH'] += ":/root/.cargo/bin"
    !git clone https://github.com/shift/FerrisRes.git /tmp/FerrisRes 2>&1 | tail -1
    !cd /tmp/FerrisRes && cargo build --release --no-default-features --features vulkan 2>&1 | tail -3
    !cp /tmp/FerrisRes/target/release/ferrisres /usr/local/bin/ferrisres
    !chmod +x /usr/local/bin/ferrisres
    # Create ICD manually for source compile fallback
    !mkdir -p /usr/share/vulkan/icd.d
    !echo '{"file_format_version":"1.0.0","ICD":{"library_path":"libGLX_nvidia.so.0","api_version":"1.3.255"}' > /usr/share/vulkan/icd.d/nvidia_icd.x86_64.json
    print("Compiled from source.")

!ferrisres --help 2>&1 | head -3

In [ ]:
# 2. Download Model + Tokenizer + Training Data
hf_model_id = f"google/gemma-4-{model_config}-it"
print(f"Downloading {hf_model_id}...")

!wget -q --show-progress https://huggingface.co/{hf_model_id}/resolve/main/model.safetensors
!wget -q https://huggingface.co/{hf_model_id}/resolve/main/tokenizer.json

# WikiText-2 training data (~2MB, ~50K tokens)
!wget -q https://raw.githubusercontent.com/pytorch/examples/master/word_language_model/data/wikitext-2/train.txt -O training_data.txt

!echo '=== Files ===' && ls -lh model.safetensors tokenizer.json training_data.txt

In [ ]:
# 3. Verify GPU Detection
!ferrisres info

In [ ]:
# 4. Run Distillation
!ferrisres distill \
  --model-path ./model.safetensors \
  --config {model_config} \
  --seq-len {seq_len} \
  --steps {max_steps} \
  --learning-rate {learning_rate} \
  --temperature 2.0 \
  --tokenizer ./tokenizer.json \
  --data training_data.txt \
  --output distilled_{model_config} \
  --log-every 50 \
  --converge {converge_threshold} \
  --converge-patience {converge_patience}

In [ ]:
# 5. (Optional) Resume if Colab session disconnected
# Uncomment and re-run to continue from last checkpoint:
# !ferrisres distill \
#   --model-path ./model.safetensors \
#   --config {model_config} \
#   --seq-len {seq_len} \
#   --steps {max_steps} \
#   --learning-rate {learning_rate} \
#   --temperature 2.0 \
#   --tokenizer ./tokenizer.json \
#   --data training_data.txt \
#   --output distilled_{model_config} \
#   --resume distilled_{model_config}.bin.checkpoint \
#   --log-every 50 \
#   --converge {converge_threshold} \
#   --converge-patience {converge_patience}

In [ ]:
# 6. Visualize Distillation Progress
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

loss_file = f'distilled_{model_config}.bin.loss_curve.csv'
try:
    df = pd.read_csv(loss_file)
    df = df.replace('n/a', np.nan)

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))

    ax1.plot(df['step'], df['kl_loss'], linewidth=2)
    ax1.set_ylabel('KL Divergence Loss')
    ax1.set_title(f'Gemma 4 {model_config.upper()} Distillation')
    ax1.grid(True, linestyle='--', alpha=0.6)

    if 'cosine_sim_avg' in df.columns:
        valid_cos = df.dropna(subset=['cosine_sim_avg'])
        valid_cos['cosine_sim_avg'] = pd.to_numeric(valid_cos['cosine_sim_avg'])
        ax2.plot(valid_cos['step'], valid_cos['cosine_sim_avg'],
                 color='darkorange', marker='o', markersize=3, alpha=0.8)
        ax2.set_ylabel('Cosine Similarity (Teacher vs Student)')
        ax2.set_xlabel('Step')
        ax2.grid(True, linestyle='--', alpha=0.6)

    plt.tight_layout()
    plt.show()
except FileNotFoundError:
    print(f'{loss_file} not found. Did the distillation complete?')